<a href="https://colab.research.google.com/github/LoLk1n-maker/recognize_text/blob/main/cnn_russian_letters.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Распознавание рукописных русских букв свёрточной сетью

Задача: картинка одной рукописной буквы → одна из 33 букв алфавита. Это MNIST, только классов не 10, а 33.

Датасет **CoMNIST**: рукописные буквы, нарисованные людьми на сенсорном экране. Каждая картинка — PNG 278×278, где буква хранится в **альфа-канале** (прозрачности).

Порядок работы:
1. Загрузка и предобработка данных
2. Свёрточная сеть
3. Обучение и оценка
4. Что видит сеть внутри: фильтры и карты признаков
5. Проверка на своей фотографии

## 1. Загрузка данных

In [ ]:
!wget -q https://github.com/GregVial/CoMNIST/raw/master/images/Cyrillic.zip
!ls -lh Cyrillic.zip

In [ ]:
import io
import zipfile

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image, ImageOps, ImageFilter

import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers

from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix

### Предобработка одной картинки

Вспомни, как устроен MNIST: светлая цифра на чёрном фоне, цифра **отцентрирована** и занимает почти всё поле. Чтобы сеть училась на похожих данных, приводим каждую букву к такому же виду:

1. Обрезаем пустые поля вокруг буквы (bounding box).
2. Кладём букву в центр квадрата с небольшим отступом, сохраняя пропорции (иначе «Ш» сплющится).
3. Уменьшаем до 28×28.

Эту же функцию потом применим к твоей фотографии: **на вход сети всегда должны приходить данные, подготовленные так же, как при обучении.**

In [ ]:
IMG_SIZE = 28

def to_square(img_l, size=IMG_SIZE, margin=0.15):
    """img_l: ч/б картинка PIL (режим 'L'), светлая буква на тёмном фоне."""
    bbox = img_l.getbbox()              # прямоугольник, где есть ненулевые пиксели
    if bbox is None:                    # пустая картинка
        return None
    img_l = img_l.crop(bbox)
    w, h = img_l.size
    side = int(max(w, h) * (1 + 2 * margin))
    canvas = Image.new("L", (side, side), 0)
    canvas.paste(img_l, ((side - w) // 2, (side - h) // 2))
    return canvas.resize((size, size), Image.LANCZOS)

### Чтение архива

Читаем картинки прямо из zip. Имена папок там кириллические, и в некоторых архивах они записаны в старой кодировке, поэтому имя при необходимости перекодируем. Папка = буква.

In [ ]:
ALPHABET = "АБВГДЕЁЖЗИЙКЛМНОПРСТУФХЦЧШЩЪЫЬЭЮЯ"

X, y = [], []
skipped = set()

with zipfile.ZipFile("Cyrillic.zip") as zf:
    for info in zf.infolist():
        name = info.filename
        if info.is_dir() or "__MACOSX" in name or not name.lower().endswith(".png"):
            continue
        if not info.flag_bits & 0x800:          # имя не в UTF-8 -> перекодируем
            try:
                name = name.encode("cp437").decode("utf-8")
            except UnicodeError:
                pass
        parts = name.split("/")
        if len(parts) < 2:
            continue
        letter = parts[-2].upper()
        if letter not in ALPHABET:
            skipped.add(parts[-2])
            continue

        img = Image.open(io.BytesIO(zf.read(info))).convert("RGBA")
        alpha = img.split()[-1]                 # буква живёт в альфа-канале
        sq = to_square(alpha)
        if sq is None:
            continue
        X.append(np.array(sq, dtype=np.uint8))
        y.append(ALPHABET.index(letter))

# оставляем только буквы, которые реально есть в архиве, и нумеруем их подряд
present = sorted(set(y))
classes = [ALPHABET[i] for i in present]
remap = {old: new for new, old in enumerate(present)}
y = np.array([remap[v] for v in y])
X = np.array(X)

print("Картинок:", X.shape, " классов:", len(classes))
print("Пропущенные папки:", skipped)
for c, n in zip(classes, np.bincount(y)):
    print(f"{c}: {n}", end="   ")

Смотрим на примеры, как ты делал в MNIST:

In [ ]:
plt.figure(figsize=(12, 6))
for k, i in enumerate(np.random.choice(len(X), 40, replace=False)):
    plt.subplot(4, 10, k + 1)
    plt.imshow(X[i], cmap=plt.cm.binary)
    plt.title(classes[y[i]])
    plt.axis("off")
plt.show()

### Подготовка к обучению

- Нормализация: делим на 255, как в MNIST.
- **Форма данных:** раньше ты делал `reshape(60000, 784)`. Для CNN картинку **не вытягиваем**: форма `(N, 28, 28, 1)`, где 1 — число каналов (ч/б).
- Делим на обучающую и тестовую выборки. `stratify=y` сохраняет одинаковую долю каждой буквы в обеих частях.

In [ ]:
X = X.astype("float32") / 255.0
X = X[..., np.newaxis]                  # (N, 28, 28) -> (N, 28, 28, 1)

x_train, x_test, y_train, y_test = train_test_split(
    X, y, test_size=0.15, stratify=y, random_state=42)

print("train:", x_train.shape, " test:", x_test.shape)

## 2. Свёрточная сеть

Архитектура ровно та, что мы разбирали в главе 15:

| Слой | Выход |
|---|---|
| Вход | 28 × 28 × 1 |
| Conv2D(32, 3×3) + ReLU | 26 × 26 × 32 |
| MaxPooling 2×2 | 13 × 13 × 32 |
| Conv2D(64, 3×3) + ReLU | 11 × 11 × 64 |
| MaxPooling 2×2 | 5 × 5 × 64 |
| Flatten | 1600 |
| Dropout(0.5) | 1600 |
| Dense(128) + ReLU | 128 |
| Dense(33) + Softmax | 33 |

Две новые вещи:

**Аугментация** (`RandomRotation`, `RandomTranslation`, `RandomZoom`). Во время обучения каждая картинка слегка поворачивается, сдвигается и масштабируется случайным образом, каждый раз по-новому. Сеть как будто видит больше разных почерков, и это защищает от переобучения. При распознавании (`predict`, `evaluate`) эти слои ничего не делают. Весов у них нет.

**`sparse_categorical_crossentropy`** вместо `categorical_crossentropy`: та же функция потерь, но правильные ответы подаются номером класса (`5`), а не one-hot вектором. Поэтому `to_categorical` не нужен.

In [ ]:
n_classes = len(classes)

model = keras.Sequential([
    keras.Input(shape=(IMG_SIZE, IMG_SIZE, 1)),

    # аугментация: работает только при обучении
    layers.RandomRotation(0.05),            # поворот до ~18 градусов
    layers.RandomTranslation(0.1, 0.1),     # сдвиг до 10%
    layers.RandomZoom(0.1),                 # масштаб +-10%

    # свёрточная часть: ищет признаки
    layers.Conv2D(32, (3, 3), activation="relu", name="conv1"),
    layers.MaxPooling2D((2, 2), name="pool1"),
    layers.Conv2D(64, (3, 3), activation="relu", name="conv2"),
    layers.MaxPooling2D((2, 2), name="pool2"),

    # полносвязная часть: по признакам решает, какая это буква
    layers.Flatten(),
    layers.Dropout(0.5),
    layers.Dense(128, activation="relu"),
    layers.Dense(n_classes, activation="softmax"),
])

model.compile(optimizer="adam",
              loss="sparse_categorical_crossentropy",
              metrics=["accuracy"])
model.summary()

Проверь в `summary()` числа параметров по формуле из главы 15.5: `(K × K × C_вход + 1) × C_выход`.

- conv1: (3·3·1 + 1) × 32 = 320
- conv2: (3·3·32 + 1) × 64 = 18 496

## 3. Обучение и оценка

In [ ]:
early_stop = keras.callbacks.EarlyStopping(
    monitor="val_loss", patience=5, restore_best_weights=True)

history = model.fit(x_train, y_train,
                    batch_size=128,
                    epochs=50,
                    validation_split=0.15,
                    callbacks=[early_stop],
                    verbose=1)

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].plot(history.history["loss"], label="обучение")
ax[0].plot(history.history["val_loss"], label="валидация")
ax[0].set_title("Потери"); ax[0].set_xlabel("эпоха"); ax[0].legend()
ax[1].plot(history.history["accuracy"], label="обучение")
ax[1].plot(history.history["val_accuracy"], label="валидация")
ax[1].set_title("Accuracy"); ax[1].set_xlabel("эпоха"); ax[1].legend()
plt.show()

Обрати внимание: из-за аугментации и Dropout accuracy на обучении может быть **ниже**, чем на валидации. Это нормально: на обучении сеть видит искажённые картинки и работает с выключенной частью нейронов.

In [ ]:
test_loss, test_acc = model.evaluate(x_test, y_test, verbose=0)
print(f"Accuracy на тесте: {test_acc * 100:.2f}%")

y_pred = np.argmax(model.predict(x_test, verbose=0), axis=1)
print(classification_report(y_test, y_pred, target_names=classes, digits=3))

Матрица ошибок 33×33 и самые частые путаницы:

In [ ]:
cm = confusion_matrix(y_test, y_pred)

plt.figure(figsize=(11, 10))
plt.imshow(cm, cmap="Blues")
plt.xticks(range(n_classes), classes)
plt.yticks(range(n_classes), classes)
plt.xlabel("Предсказание сети"); plt.ylabel("Правильная буква")
plt.colorbar()
plt.show()

off = cm.copy()
np.fill_diagonal(off, 0)
order = np.argsort(off.ravel())[::-1][:10]
print("Чаще всего путает:")
for flat in order:
    t, p = np.unravel_index(flat, off.shape)
    if off[t, p] == 0:
        break
    print(f"  {classes[t]} -> {classes[p]}: {off[t, p]} раз")

Посмотрим на ошибки глазами: часто там и человек засомневается.

In [ ]:
wrong = np.where(y_pred != y_test)[0]
plt.figure(figsize=(12, 5))
for k, i in enumerate(wrong[:20]):
    plt.subplot(2, 10, k + 1)
    plt.imshow(x_test[i, :, :, 0], cmap=plt.cm.binary)
    plt.title(f"{classes[y_test[i]]}->{classes[y_pred[i]]}")
    plt.axis("off")
plt.show()

## 4. Что видит сеть внутри

Здесь можно своими глазами увидеть разницу между **фильтрами** (веса) и **картами признаков** (данные), которую мы разбирали.

### Фильтры первого слоя

Это 32 таблички 3×3 — **веса**, которые сеть выучила. Они одинаковы для всех картинок.

In [ ]:
w = model.get_layer("conv1").get_weights()[0]     # форма (3, 3, 1, 32)
print("Форма весов conv1:", w.shape)

plt.figure(figsize=(10, 5))
for k in range(32):
    plt.subplot(4, 8, k + 1)
    plt.imshow(w[:, :, 0, k], cmap="gray")
    plt.axis("off")
plt.suptitle("32 фильтра первого слоя (светлое = большой вес)")
plt.show()

### Карты признаков для одной буквы

А это **данные**: что получилось, когда эти 32 фильтра прошли по конкретной букве. Для другой буквы карты будут другими, а фильтры останутся теми же.

In [ ]:
i = 0
sample = x_test[i:i + 1]                           # батч из одной картинки
print("Буква:", classes[y_test[i]])

conv1_out = keras.Model(model.inputs, model.get_layer("conv1").output)
conv2_out = keras.Model(model.inputs, model.get_layer("conv2").output)

maps1 = conv1_out.predict(sample, verbose=0)[0]    # (26, 26, 32)
maps2 = conv2_out.predict(sample, verbose=0)[0]    # (11, 11, 64)
print("Карты conv1:", maps1.shape, "  карты conv2:", maps2.shape)

plt.figure(figsize=(3, 3))
plt.imshow(sample[0, :, :, 0], cmap=plt.cm.binary); plt.axis("off")
plt.show()

for maps, title in [(maps1, "conv1: 32 карты 26×26 (штрихи, края)"),
                    (maps2, "conv2: первые 32 из 64 карт 11×11 (комбинации)")]:
    plt.figure(figsize=(12, 6))
    for k in range(32):
        plt.subplot(4, 8, k + 1)
        plt.imshow(maps[:, :, k], cmap="viridis")
        plt.axis("off")
    plt.suptitle(title)
    plt.show()

## 5. Проверка на своей фотографии

Напиши одну букву на белой бумаге **толстым маркером или фломастером**, крупно, и сфотографируй. Тонкая ручка после уменьшения до 28×28 почти исчезнет.

Предобработка повторяет то, что было при обучении: светлая буква на тёмном фоне, обрезка, центрирование. Плюс утолщение линий, потому что в датасете буквы рисовали пальцем.

In [ ]:
def prepare_photo(path):
    img = Image.open(path).convert("L")
    img = ImageOps.autocontrast(img)
    if np.array(img).mean() > 127:                  # тёмная буква на светлом фоне
        img = ImageOps.invert(img)                  # -> светлая на тёмном, как в датасете
    img = img.point(lambda v: 0 if v < 100 else v)  # убираем серый фон бумаги
    bbox = img.getbbox()
    if bbox is None:
        raise ValueError("Не нашёл букву на картинке")
    img = img.crop(bbox)
    k = max(3, (max(img.size) // 25) | 1)           # нечётный размер для утолщения
    img = img.filter(ImageFilter.MaxFilter(k))      # утолщаем линии
    return to_square(img)

from google.colab import files
uploaded = files.upload()
path = next(iter(uploaded))

sq = prepare_photo(path)
x = np.array(sq, dtype="float32")[np.newaxis, ..., np.newaxis] / 255.0   # (1, 28, 28, 1)

probs = model.predict(x, verbose=0)[0]
top = np.argsort(probs)[::-1][:3]

plt.figure(figsize=(3, 3))
plt.imshow(x[0, :, :, 0], cmap=plt.cm.binary); plt.axis("off")
plt.title("Что видит сеть")
plt.show()

for r, c in enumerate(top, 1):
    print(f"{r}. {classes[c]}  ({probs[c] * 100:.1f}%)")

## Что попробовать дальше

1. Убери слои аугментации и сравни кривые обучения и accuracy на тесте.
2. Добавь третий блок `Conv2D(128, 3×3, padding="same")` + `MaxPooling` и посчитай руками, какие получатся размеры.
3. Замени `Dropout(0.5)` на `Dropout(0.2)` или убери его: как изменится разрыв между обучением и валидацией?
4. Посмотри на самые частые путаницы: какие пары букв сложнее всего и почему?